# Synthetic Photometry — documentation examples

Companion notebook to the **Synthetic Photometry** documentation page
(`docs/source/synthetic_photometry.rst`). One section per code snippet; the
printed values are the ones quoted on the page.

Requires only the core `stellar-spice` install.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import warnings
import numpy as np
import jax.numpy as jnp

## Passband magnitudes from a spectrum

A solar blackbody at 10 pc, so the AB magnitudes are absolute magnitudes.

In [2]:
from spice.models import IcosphereModel
from spice.spectrum import Blackbody, simulate_observed_flux
from spice.spectrum import (AB_passband_luminosity, ST_passband_luminosity,
                            Vega_passband_luminosity)
from spice.spectrum.filter import JohnsonCousinsV, GaiaG

bb = Blackbody()
star = IcosphereModel.construct(100, 1., 1.,
                                bb.solar_parameters, bb.parameter_names)
wavelengths = jnp.linspace(3000., 12000., 2001)  # Angstroms
spectrum = simulate_observed_flux(bb.intensity, star,
                                 jnp.log10(wavelengths), distance=10.)
flux = spectrum[:, 0]  # erg/s/cm²/Angstrom, not the full two-column array

magnitudes = {
    'V_AB': AB_passband_luminosity(JohnsonCousinsV(), wavelengths, flux),
    'V_ST': ST_passband_luminosity(JohnsonCousinsV(), wavelengths, flux),
    'G_AB': AB_passband_luminosity(GaiaG(), wavelengths, flux),
    'G_Vega': Vega_passband_luminosity(GaiaG(), wavelengths, flux),
}
for name, mag in magnitudes.items():
    print(f"{name:7s} {float(mag):7.3f} mag")

[spice] IcosphereModel constructed in 0.8 s


V_AB      4.864 mag
V_ST      4.869 mag
G_AB      4.809 mag
G_Vega    4.695 mag


Distance scaling: moving the same star from 10 pc to 100 pc changes every
magnitude by $5\log_{10}(100/10) = 5$ mag.

In [3]:
flux_100pc = simulate_observed_flux(bb.intensity, star, jnp.log10(wavelengths),
                                    distance=100.)[:, 0]
dV = AB_passband_luminosity(JohnsonCousinsV(), wavelengths, flux_100pc) - magnitudes['V_AB']
print(f"V(100 pc) - V(10 pc) = {float(dV):.4f} mag")

V(100 pc) - V(10 pc) = 5.0000 mag


## Wavelength coverage

Fraction of each passband's response inside the HARPS intensity emulator's
3780–6910 Å, and the warning raised for a partly covered passband.

In [4]:
from spice.spectrum import WavelengthCoverageWarning
from spice.spectrum.filter import HipparcosHp, TWOMASSJ

harps = jnp.linspace(3780., 6910., 4000)
for f in [HipparcosHp(), GaiaG(), TWOMASSJ()]:
    print(f"{f.name:12s} covered fraction: {float(f.wavelength_coverage(harps)):.2f}")

harps_flux = simulate_observed_flux(bb.intensity, star, jnp.log10(harps))[:, 0]
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    AB_passband_luminosity(HipparcosHp(), harps, harps_flux)
for w in caught:
    if issubclass(w.category, WavelengthCoverageWarning):
        print(f"{w.category.__name__}: {w.message}")

Hipparcos Hp covered fraction: 0.91
Gaia G       covered fraction: 0.61
2MASS J      covered fraction: 0.00


Share of the bolometric flux of a 4500–8000 K blackbody inside 3780–6910 Å
(the page quotes 27–41 per cent).

In [5]:
from spice.spectrum.blackbody import blackbody_intensity

wl_all = np.geomspace(100., 1.e6, 20000)
inside = (wl_all >= 3780.) & (wl_all <= 6910.)
for teff in [4500., 8000.]:
    b = np.asarray(blackbody_intensity(np.log10(wl_all), 1., jnp.array([teff]))[:, 0])
    print(f"Teff {teff:.0f} K: {np.trapezoid(b * inside, wl_all) / np.trapezoid(b, wl_all):.2f}")

Teff 4500 K: 0.27
Teff 8000 K: 0.41


## Reference spectra and custom filters

In [6]:
from spice.spectrum.filter import Filter

reference_filter = Filter(
    jnp.array([[4000., 4500., 6000., 7000.], [0., 0.8, 1., 0.]]),
    name='Reference passband', non_photonic=False,
)
# c in Angstrom/s; F_nu in erg/s/cm²/Hz.
ab_reference = 3.631e-20 * 2.99792458e18 / wavelengths**2
ab_zero = AB_passband_luminosity(reference_filter, wavelengths, ab_reference)

st_reference = jnp.full_like(wavelengths, 10.**(-21.10 / 2.5))
st_zero = ST_passband_luminosity(reference_filter, wavelengths, st_reference)
print(float(ab_zero), float(st_zero))  # approximately 0, 0 mag

3.5546839870531175e-16 4.160566451444734e-15


## Luminosity and mesh convergence

`Blackbody.flux` returns the `mu=1` intensity, so the emergent surface flux
of the un-darkened blackbody is $\pi$ times that intensity.

In [7]:
from spice.spectrum import luminosity, absolute_bol_luminosity

def surface_flux(log_wavelengths, parameters):
    return jnp.pi * bb.intensity(log_wavelengths, 1., parameters)

lum_wavelengths = np.geomspace(100., 1.e6, 4096)  # Angstroms
L = luminosity(surface_flux, star, lum_wavelengths)  # erg/s
M_bol = absolute_bol_luminosity(L)

# Stefan-Boltzmann prediction using this mesh's surface area.
sigma_cgs = 5.670374419e-5  # erg/s/cm²/K⁴
surface_area_cm2 = jnp.sum(star.areas) * (star.radius * 6.957e10)**2
L_stefan_boltzmann = surface_area_cm2 * sigma_cgs * bb.solar_parameters[0]**4
print(float(L / L_stefan_boltzmann))  # approximately 1
print(f"L = {float(L):.4e} erg/s, M_bol = {float(M_bol):.3f} mag")

1.0000000575866268
L = 3.8565e+33 erg/s, M_bol = 4.732 mag


Mesh convergence: the mesh area approaches $4\pi R^2$, so the luminosity
approaches the Stefan-Boltzmann value for a sphere of radius $R$.

In [8]:
L_sphere = 4 * np.pi * (6.957e10)**2 * sigma_cgs * float(bb.solar_parameters[0])**4
for n in [100, 1000, 5000]:
    mesh = IcosphereModel.construct(n, 1., 1., bb.solar_parameters, bb.parameter_names)
    L_n = luminosity(surface_flux, mesh, lum_wavelengths)
    print(f"{len(mesh.d_vertices):5d} vertices: area / 4 pi R^2 = "
          f"{float(jnp.sum(mesh.areas)) / (4 * np.pi):.5f}, L / L_sphere = {float(L_n) / L_sphere:.5f}")

[spice] IcosphereModel constructed in 0.0 s


  312 vertices: area / 4 pi R^2 = 1.00396, L / L_sphere = 1.00396


[spice] IcosphereModel constructed in 0.4 s


 1272 vertices: area / 4 pi R^2 = 1.00099, L / L_sphere = 1.00099


[spice] IcosphereModel constructed in 0.5 s


 5112 vertices: area / 4 pi R^2 = 1.00025, L / L_sphere = 1.00025
